In [1]:
from __future__ import annotations

import math
import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, GeoCoordinate, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "EvChargingStation"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

stations = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="operator",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="city_area",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="charger_type",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="power_kw",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="available",
            data_type=wvc.config.DataType.BOOL,
        ),
        wvc.config.Property(
            name="open_24h",
            data_type=wvc.config.DataType.BOOL,
        ),
        wvc.config.Property(
            name="price_per_kwh",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="location",
            data_type=wvc.config.DataType.GEO_COORDINATES,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: EvChargingStation


In [5]:
station_data = [
    {
        "name": "Central Fast Charge",
        "description": "Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.",
        "operator": "VoltWay",
        "city_area": "Centrum",
        "charger_type": "DC fast",
        "power_kw": 150,
        "available": True,
        "open_24h": True,
        "price_per_kwh": 2.40,
        "location": {
            "latitude": 52.2297,
            "longitude": 21.0122,
        },
    },
    {
        "name": "Wola Business Charger",
        "description": "Charging station close to office buildings and business parking areas.",
        "operator": "GreenPlug",
        "city_area": "Wola",
        "charger_type": "DC fast",
        "power_kw": 120,
        "available": True,
        "open_24h": False,
        "price_per_kwh": 2.10,
        "location": {
            "latitude": 52.2362,
            "longitude": 20.9606,
        },
    },
    {
        "name": "Mokotow Office Charge",
        "description": "Reliable charging point for office workers and commuters in the Mokotow business district.",
        "operator": "VoltWay",
        "city_area": "Mokotow",
        "charger_type": "AC",
        "power_kw": 22,
        "available": True,
        "open_24h": False,
        "price_per_kwh": 1.65,
        "location": {
            "latitude": 52.1846,
            "longitude": 21.0085,
        },
    },
    {
        "name": "Praga Night Charger",
        "description": "24-hour charging station on the eastern side of the city, useful for late evening charging.",
        "operator": "EcoCharge",
        "city_area": "Praga",
        "charger_type": "DC fast",
        "power_kw": 100,
        "available": True,
        "open_24h": True,
        "price_per_kwh": 2.25,
        "location": {
            "latitude": 52.2540,
            "longitude": 21.0347,
        },
    },
    {
        "name": "Ursynow Residential Charger",
        "description": "AC charging station near residential buildings, good for longer parking sessions.",
        "operator": "GreenPlug",
        "city_area": "Ursynow",
        "charger_type": "AC",
        "power_kw": 11,
        "available": True,
        "open_24h": True,
        "price_per_kwh": 1.35,
        "location": {
            "latitude": 52.1410,
            "longitude": 21.0500,
        },
    },
    {
        "name": "Airport Express Charger",
        "description": "High power charger near the airport, suitable for quick charging before or after travel.",
        "operator": "VoltWay",
        "city_area": "Airport",
        "charger_type": "DC ultra-fast",
        "power_kw": 250,
        "available": False,
        "open_24h": True,
        "price_per_kwh": 2.80,
        "location": {
            "latitude": 52.1657,
            "longitude": 20.9671,
        },
    },
    {
        "name": "Bielany Park Charger",
        "description": "Quiet charging point near parks and residential areas, better for relaxed longer charging.",
        "operator": "EcoCharge",
        "city_area": "Bielany",
        "charger_type": "AC",
        "power_kw": 22,
        "available": True,
        "open_24h": False,
        "price_per_kwh": 1.50,
        "location": {
            "latitude": 52.2943,
            "longitude": 20.9290,
        },
    },
    {
        "name": "Wilnow Premium Charger",
        "description": "Premium charging station near shopping and residential zones, good for medium-length stops.",
        "operator": "GreenPlug",
        "city_area": "Wilanow",
        "charger_type": "DC fast",
        "power_kw": 80,
        "available": True,
        "open_24h": False,
        "price_per_kwh": 2.00,
        "location": {
            "latitude": 52.1651,
            "longitude": 21.0903,
        },
    },
]

In [6]:
result = stations.data.insert_many(station_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = stations.query.fetch_objects(
    limit=20,
    return_properties=[
        "name",
        "operator",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
        "location",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 01bbfad6-2807-446e-a1bf-783036cf2117
{'operator': 'GreenPlug', 'open_24h': False, 'power_kw': 120, 'price_per_kwh': 2.1, 'location': GeoCoordinate(latitude=52.23619842529297, longitude=20.960599899291992), 'available': True, 'name': 'Wola Business Charger', 'city_area': 'Wola', 'charger_type': 'DC fast'}
--------------------------------------------------------------------------------
UUID: 0c9cbf57-4388-4aa0-be15-44e57b3ce3b0
{'operator': 'EcoCharge', 'open_24h': True, 'power_kw': 100, 'price_per_kwh': 2.25, 'location': GeoCoordinate(latitude=52.25400161743164, longitude=21.034700393676758), 'available': True, 'name': 'Praga Night Charger', 'city_area': 'Praga', 'charger_type': 'DC fast'}
--------------------------------------------------------------------------------
UUID: 2bb2ece1-cc01-40a3-906a-c26416123745
{'operator': 'VoltWay', 'open_24h': False, 'power_kw': 22, 'price_per_kwh': 1.65, 'location': GeoCoordinate(latitude=52.184600830078125, longitude=21.008499145507812), 'ava

In [8]:
warsaw_center = GeoCoordinate(
    latitude=52.2297,
    longitude=21.0122,
)

response = stations.query.fetch_objects(
    filters=Filter.by_property("location").within_geo_range(
        coordinate=warsaw_center,
        distance=5000,
    ),
    limit=20,
    return_properties=[
        "name",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
        "location",
    ],
)

for obj in response.objects:
    print("Name:", obj.properties["name"])
    print("Area:", obj.properties["city_area"])
    print("Type:", obj.properties["charger_type"])
    print("Power:", obj.properties["power_kw"])
    print("Available:", obj.properties["available"])
    print("Location:", obj.properties["location"])
    print("-" * 80)

Name: Wola Business Charger
Area: Wola
Type: DC fast
Power: 120
Available: True
Location: latitude=52.23619842529297 longitude=20.960599899291992
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Type: DC fast
Power: 150
Available: True
Location: latitude=52.229698181152344 longitude=21.01219940185547
--------------------------------------------------------------------------------
Name: Praga Night Charger
Area: Praga
Type: DC fast
Power: 100
Available: True
Location: latitude=52.25400161743164 longitude=21.034700393676758
--------------------------------------------------------------------------------


In [9]:
filters = (
    Filter.by_property("location").within_geo_range(
        coordinate=warsaw_center,
        distance=8000,
    )
    & Filter.by_property("available").equal(True)
)

response = stations.query.fetch_objects(
    filters=filters,
    limit=20,
    return_properties=[
        "name",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
    ],
)

for obj in response.objects:
    print("Name:", obj.properties["name"])
    print("Area:", obj.properties["city_area"])
    print("Type:", obj.properties["charger_type"])
    print("Power:", obj.properties["power_kw"])
    print("Price:", obj.properties["price_per_kwh"])
    print("-" * 80)

Name: Wola Business Charger
Area: Wola
Type: DC fast
Power: 120
Price: 2.1
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Type: DC fast
Power: 150
Price: 2.4
--------------------------------------------------------------------------------
Name: Mokotow Office Charge
Area: Mokotow
Type: AC
Power: 22
Price: 1.65
--------------------------------------------------------------------------------
Name: Praga Night Charger
Area: Praga
Type: DC fast
Power: 100
Price: 2.25
--------------------------------------------------------------------------------


In [10]:
filters = (
    Filter.by_property("location").within_geo_range(
        coordinate=warsaw_center,
        distance=10000,
    )
    & Filter.by_property("available").equal(True)
    & Filter.by_property("power_kw").greater_or_equal(100)
)

response = stations.query.fetch_objects(
    filters=filters,
    limit=20,
    return_properties=[
        "name",
        "description",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
        "location",
    ],
)

for obj in response.objects:
    print("Name:", obj.properties["name"])
    print("Area:", obj.properties["city_area"])
    print("Type:", obj.properties["charger_type"])
    print("Power:", obj.properties["power_kw"])
    print("Price:", obj.properties["price_per_kwh"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Name: Wola Business Charger
Area: Wola
Type: DC fast
Power: 120
Price: 2.1
Description: Charging station close to office buildings and business parking areas.
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Type: DC fast
Power: 150
Price: 2.4
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Name: Praga Night Charger
Area: Praga
Type: DC fast
Power: 100
Price: 2.25
Description: 24-hour charging station on the eastern side of the city, useful for late evening charging.
--------------------------------------------------------------------------------


In [11]:
geo_filter = Filter.by_property("location").within_geo_range(
    coordinate=warsaw_center,
    distance=12000,
)

response = stations.query.near_text(
    query="fast charger for quick top-up near business area",
    filters=geo_filter,
    limit=5,
    return_properties=[
        "name",
        "description",
        "operator",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "price_per_kwh",
        "location",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Vector distance:", obj.metadata.distance)
    print("Name:", obj.properties["name"])
    print("Area:", obj.properties["city_area"])
    print("Power:", obj.properties["power_kw"])
    print("Available:", obj.properties["available"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Vector distance: 0.3529844284057617
Name: Central Fast Charge
Area: Centrum
Power: 150
Available: True
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Vector distance: 0.3765517473220825
Name: Wola Business Charger
Area: Wola
Power: 120
Available: True
Description: Charging station close to office buildings and business parking areas.
--------------------------------------------------------------------------------
Vector distance: 0.40271127223968506
Name: Airport Express Charger
Area: Airport
Power: 250
Available: False
Description: High power charger near the airport, suitable for quick charging before or after travel.
--------------------------------------------------------------------------------
Vector distance: 0.42400527000427246
Name: Wilnow Premium Charger
Area: Wilanow
Power: 80
Available: True
Description: Premium charging station n

In [12]:
filters = (
    Filter.by_property("location").within_geo_range(
        coordinate=warsaw_center,
        distance=15000,
    )
    & Filter.by_property("available").equal(True)
    & Filter.by_property("open_24h").equal(True)
)

response = stations.query.hybrid(
    query="night fast charger open 24h",
    alpha=0.5,
    filters=filters,
    limit=5,
    return_properties=[
        "name",
        "description",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
        "location",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Hybrid score:", obj.metadata.score)
    print("Name:", obj.properties["name"])
    print("Area:", obj.properties["city_area"])
    print("Type:", obj.properties["charger_type"])
    print("Power:", obj.properties["power_kw"])
    print("Open 24h:", obj.properties["open_24h"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Hybrid score: 1.0
Name: Praga Night Charger
Area: Praga
Type: DC fast
Power: 100
Open 24h: True
Description: 24-hour charging station on the eastern side of the city, useful for late evening charging.
--------------------------------------------------------------------------------
Hybrid score: 0.6920607089996338
Name: Central Fast Charge
Area: Centrum
Type: DC fast
Power: 150
Open 24h: True
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Hybrid score: 0.0
Name: Ursynow Residential Charger
Area: Ursynow
Type: AC
Power: 11
Open 24h: True
Description: AC charging station near residential buildings, good for longer parking sessions.
--------------------------------------------------------------------------------


In [13]:
def haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    earth_radius_km = 6371.0

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    a = (
        math.sin(delta_phi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return earth_radius_km * c

In [17]:
def extract_location(location) -> tuple[float, float]:
    if isinstance(location, dict):
        return location["latitude"], location["longitude"]

    return location.latitude, location.longitude

In [18]:
def find_nearby_chargers(
    *,
    latitude: float,
    longitude: float,
    radius_meters: int,
    query: str,
    min_power_kw: int = 0,
    only_available: bool = True,
    open_24h: bool | None = None,
    limit: int = 10,
) -> None:
    user_coordinate = GeoCoordinate(
        latitude=latitude,
        longitude=longitude,
    )

    filters = Filter.by_property("location").within_geo_range(
        coordinate=user_coordinate,
        distance=radius_meters,
    )

    if only_available:
        filters = filters & Filter.by_property("available").equal(True)

    if min_power_kw > 0:
        filters = filters & Filter.by_property("power_kw").greater_or_equal(min_power_kw)

    if open_24h is not None:
        filters = filters & Filter.by_property("open_24h").equal(open_24h)

    response = stations.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "name",
            "description",
            "operator",
            "city_area",
            "charger_type",
            "power_kw",
            "available",
            "open_24h",
            "price_per_kwh",
            "location",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    results = []

    for obj in response.objects:
        station_lat, station_lon = extract_location(obj.properties["location"])

        distance_km = haversine_km(
            latitude,
            longitude,
            station_lat,
            station_lon,
        )

        results.append(
            {
                "name": obj.properties["name"],
                "area": obj.properties["city_area"],
                "operator": obj.properties["operator"],
                "charger_type": obj.properties["charger_type"],
                "power_kw": obj.properties["power_kw"],
                "available": obj.properties["available"],
                "open_24h": obj.properties["open_24h"],
                "price_per_kwh": obj.properties["price_per_kwh"],
                "description": obj.properties["description"],
                "vector_distance": obj.metadata.distance,
                "distance_km": round(distance_km, 2),
            }
        )

    results = sorted(results, key=lambda item: item["distance_km"])

    print("QUERY:", query)
    print("RADIUS METERS:", radius_meters)
    print("MIN POWER:", min_power_kw)
    print("ONLY AVAILABLE:", only_available)
    print("OPEN 24H:", open_24h)
    print("-" * 80)

    if not results:
        print("No matching chargers found.")
        return

    for item in results:
        print("Name:", item["name"])
        print("Area:", item["area"])
        print("Distance km:", item["distance_km"])
        print("Vector distance:", item["vector_distance"])
        print("Type:", item["charger_type"])
        print("Power:", item["power_kw"])
        print("Open 24h:", item["open_24h"])
        print("Price:", item["price_per_kwh"])
        print("Description:", item["description"])
        print("-" * 80)

In [19]:
find_nearby_chargers(
    latitude=52.2297,
    longitude=21.0122,
    radius_meters=10000,
    query="fast charger for quick charging",
    min_power_kw=100,
    only_available=True,
)

QUERY: fast charger for quick charging
RADIUS METERS: 10000
MIN POWER: 100
ONLY AVAILABLE: True
OPEN 24H: None
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Distance km: 0.0
Vector distance: 0.4511014223098755
Type: DC fast
Power: 150
Open 24h: True
Price: 2.4
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Name: Praga Night Charger
Area: Praga
Distance km: 3.11
Vector distance: 0.5141215920448303
Type: DC fast
Power: 100
Open 24h: True
Price: 2.25
Description: 24-hour charging station on the eastern side of the city, useful for late evening charging.
--------------------------------------------------------------------------------
Name: Wola Business Charger
Area: Wola
Distance km: 3.59
Vector distance: 0.5158551335334778
Type: DC fast
Power: 120
Open 24h: False
Price: 

In [20]:
find_nearby_chargers(
    latitude=52.2297,
    longitude=21.0122,
    radius_meters=15000,
    query="night charger open all day",
    min_power_kw=50,
    only_available=True,
    open_24h=True,
)

QUERY: night charger open all day
RADIUS METERS: 15000
MIN POWER: 50
ONLY AVAILABLE: True
OPEN 24H: True
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Distance km: 0.0
Vector distance: 0.60927414894104
Type: DC fast
Power: 150
Open 24h: True
Price: 2.4
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Name: Praga Night Charger
Area: Praga
Distance km: 3.11
Vector distance: 0.49844640493392944
Type: DC fast
Power: 100
Open 24h: True
Price: 2.25
Description: 24-hour charging station on the eastern side of the city, useful for late evening charging.
--------------------------------------------------------------------------------


In [21]:
find_nearby_chargers(
    latitude=52.1846,
    longitude=21.0085,
    radius_meters=12000,
    query="cheap AC charger for longer parking",
    min_power_kw=0,
    only_available=True,
)

QUERY: cheap AC charger for longer parking
RADIUS METERS: 12000
MIN POWER: 0
ONLY AVAILABLE: True
OPEN 24H: None
--------------------------------------------------------------------------------
Name: Mokotow Office Charge
Area: Mokotow
Distance km: 0.0
Vector distance: 0.5201345086097717
Type: AC
Power: 22
Open 24h: False
Price: 1.65
Description: Reliable charging point for office workers and commuters in the Mokotow business district.
--------------------------------------------------------------------------------
Name: Central Fast Charge
Area: Centrum
Distance km: 5.02
Vector distance: 0.5611190795898438
Type: DC fast
Power: 150
Open 24h: True
Price: 2.4
Description: Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.
--------------------------------------------------------------------------------
Name: Ursynow Residential Charger
Area: Ursynow
Distance km: 5.61
Vector distance: 0.3418735861778259
Type: AC
Power: 11
Open 24h: True
Price

In [22]:
filters = (
    Filter.by_property("location").within_geo_range(
        coordinate=warsaw_center,
        distance=12000,
    )
    & Filter.by_property("available").equal(True)
    & Filter.by_property("power_kw").greater_or_equal(80)
)

response = stations.generate.near_text(
    query="best available fast charger near the center for quick charging",
    filters=filters,
    grouped_task=(
        "Recommend the best charging station from the retrieved results. "
        "Use only the retrieved station data. "
        "Mention area, charger type, power, availability and price. "
        "Keep the answer concise and practical."
    ),
    limit=5,
    return_properties=[
        "name",
        "description",
        "operator",
        "city_area",
        "charger_type",
        "power_kw",
        "available",
        "open_24h",
        "price_per_kwh",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print(
        "-",
        obj.properties["name"],
        "|",
        obj.properties["city_area"],
        "|",
        obj.properties["power_kw"],
        "kW",
    )

The best charging station based on the retrieved data is:

**Name**: Central Fast Charge  
- **Area**: Centrum  
- **Charger Type**: DC fast  
- **Power**: 150 kW  
- **Availability**: Open 24 hours, available  
- **Price**: €2.4 per kWh  

This station is suitable for quick top-ups during work or shopping near the city center, making it a convenient choice.

Sources:
- Central Fast Charge | Centrum | 150 kW
- Wilnow Premium Charger | Wilanow | 80 kW
- Wola Business Charger | Wola | 120 kW
- Praga Night Charger | Praga | 100 kW


In [23]:
def recommend_charger(
    *,
    latitude: float,
    longitude: float,
    radius_meters: int,
    user_need: str,
    min_power_kw: int = 0,
    only_available: bool = True,
    open_24h: bool | None = None,
    limit: int = 5,
) -> None:
    user_coordinate = GeoCoordinate(
        latitude=latitude,
        longitude=longitude,
    )

    filters = Filter.by_property("location").within_geo_range(
        coordinate=user_coordinate,
        distance=radius_meters,
    )

    if only_available:
        filters = filters & Filter.by_property("available").equal(True)

    if min_power_kw > 0:
        filters = filters & Filter.by_property("power_kw").greater_or_equal(min_power_kw)

    if open_24h is not None:
        filters = filters & Filter.by_property("open_24h").equal(open_24h)

    response = stations.generate.near_text(
        query=user_need,
        filters=filters,
        grouped_task=(
            "Act as an EV charging recommendation assistant. "
            "Use only the retrieved charging stations. "
            "Recommend the most suitable station. "
            "Mention area, charger type, power, availability, 24h status and price. "
            "If there is no clearly suitable station, say so."
        ),
        limit=limit,
        return_properties=[
            "name",
            "description",
            "operator",
            "city_area",
            "charger_type",
            "power_kw",
            "available",
            "open_24h",
            "price_per_kwh",
            "location",
        ],
    )

    print("USER NEED:", user_need)
    print("RADIUS:", radius_meters)
    print("MIN POWER:", min_power_kw)
    print("ONLY AVAILABLE:", only_available)
    print("OPEN 24H:", open_24h)
    print("-" * 80)

    print(response.generated)

    print("\nRetrieved stations:")
    for obj in response.objects:
        station_lat, station_lon = extract_location(obj.properties["location"])

        distance_km = haversine_km(
            latitude,
            longitude,
            station_lat,
            station_lon,
        )

        print(
            "-",
            obj.properties["name"],
            "|",
            obj.properties["city_area"],
            "|",
            obj.properties["power_kw"],
            "kW |",
            round(distance_km, 2),
            "km",
        )

In [24]:
recommend_charger(
    latitude=52.2297,
    longitude=21.0122,
    radius_meters=12000,
    user_need="I need a fast charger for a quick top-up near the city center",
    min_power_kw=80,
    only_available=True,
)

USER NEED: I need a fast charger for a quick top-up near the city center
RADIUS: 12000
MIN POWER: 80
ONLY AVAILABLE: True
OPEN 24H: None
--------------------------------------------------------------------------------
Based on the retrieved charging stations, I recommend the **Central Fast Charge** station located in the **Centrum** area. Here are the details:

- **Charger Type:** DC Fast
- **Power:** 150 kW
- **Availability:** True
- **24h Status:** Open 24 hours
- **Price per kWh:** 2.4 PLN

This station is ideal for quick top-ups during work or shopping, making it a convenient choice for those in the city center.

Retrieved stations:
- Central Fast Charge | Centrum | 150 kW | 0.0 km
- Praga Night Charger | Praga | 100 kW | 3.11 km
- Wilnow Premium Charger | Wilanow | 80 kW | 8.94 km
- Wola Business Charger | Wola | 120 kW | 3.59 km


In [25]:
recommend_charger(
    latitude=52.2540,
    longitude=21.0347,
    radius_meters=10000,
    user_need="I need an available charger open at night",
    min_power_kw=50,
    only_available=True,
    open_24h=True,
)

USER NEED: I need an available charger open at night
RADIUS: 10000
MIN POWER: 50
ONLY AVAILABLE: True
OPEN 24H: True
--------------------------------------------------------------------------------
Based on the retrieved charging stations, I recommend the following:

### Central Fast Charge
- **Area:** Centrum
- **Charger Type:** DC fast
- **Power:** 150 kW
- **Availability:** Available
- **24-Hour Status:** Open 24 hours
- **Price:** €2.40 per kWh
- **Description:** Fast EV charging station near the city center, suitable for quick top-ups during work or shopping.

This station is ideal for a convenient and fast charging experience, especially if you're in the city center.

Retrieved stations:
- Praga Night Charger | Praga | 100 kW | 0.0 km
- Central Fast Charge | Centrum | 150 kW | 3.11 km


In [26]:
recommend_charger(
    latitude=52.1846,
    longitude=21.0085,
    radius_meters=15000,
    user_need="I want a cheaper charger for longer parking near an office area",
    min_power_kw=0,
    only_available=True,
)

USER NEED: I want a cheaper charger for longer parking near an office area
RADIUS: 15000
MIN POWER: 0
ONLY AVAILABLE: True
OPEN 24H: None
--------------------------------------------------------------------------------
Based on the retrieved charging stations, I recommend the **Praga Night Charger**. Here are the details:

- **Area**: Praga
- **Charger Type**: DC fast
- **Power**: 100 kW
- **Availability**: Available
- **24h Status**: Open 24 hours
- **Price**: 2.25 PLN per kWh

This station is ideal for late evening charging, making it very convenient if you need to charge your EV at any time of the day or night.

Retrieved stations:
- Wola Business Charger | Wola | 120 kW | 6.6 km
- Ursynow Residential Charger | Ursynow | 11 kW | 5.61 km
- Bielany Park Charger | Bielany | 22 kW | 13.35 km
- Wilnow Premium Charger | Wilanow | 80 kW | 5.98 km
- Praga Night Charger | Praga | 100 kW | 7.92 km


In [27]:
client.close()